In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np
import gdown
import re
from collections import Counter

# 1. Загрузка данных
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l7/20writers.zip', None, quiet=True)
!unzip -qo 20writers.zip -d writers/

# 2. Параметры
MAX_WORDS = 20000
X_LEN = 150
BATCH_SIZE = 128
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def load_data(path):
    files = sorted(os.listdir(path))
    texts = []
    for f in files:
        with open(os.path.join(path, f), 'r', errors='ignore') as file:
            texts.append(file.read().replace('\n', ' ').lower())
    return texts, [f.replace('.txt', '') for f in files]

all_texts, class_names = load_data('writers/')
NUM_CLASSES = len(class_names)

# 3. Токенизация (простой Vocab)
all_words = re.findall(r'\w+', ' '.join(all_texts))
vocab = {word: i+2 for i, (word, _) in enumerate(Counter(all_words).most_common(MAX_WORDS-2))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

def text_to_seq(text):
    return [vocab.get(w, 1) for w in re.findall(r'\w+', text)]

# Разделение 80/20
train_seqs = [text_to_seq(t[:int(len(t)*0.8)]) for t in all_texts]
val_seqs = [text_to_seq(t[int(len(t)*0.8):]) for t in all_texts]

In [ ]:
class WriterDataset(Dataset):
    def __init__(self, sequences, x_len, samples_per_class):
        self.data = []
        self.labels = []
        for i, seq in enumerate(sequences):
            step = max(1, (len(seq) - x_len) // samples_per_class)
            count = 0
            for j in range(0, len(seq) - x_len + 1, step):
                if count < samples_per_class:
                    self.data.append(torch.LongTensor(seq[j:j+x_len]))
                    self.labels.append(i)
                    count += 1

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

train_loader = DataLoader(WriterDataset(train_seqs, X_LEN, 2000), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(WriterDataset(val_seqs, X_LEN, 400), batch_size=BATCH_SIZE)

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.spatial_dropout = nn.Dropout2d(0.2)
        self.conv = nn.Conv1d(emb_dim, 128, kernel_size=5)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # x: (batch, seq_len)
        x = self.embedding(x) # (batch, seq_len, emb_dim)
        x = x.permute(0, 2, 1) # (batch, emb_dim, seq_len) для Conv1d
        x = self.spatial_dropout(x.unsqueeze(2)).squeeze(2)
        x = self.conv(x)
        x = self.relu(x)
        x = self.pool(x).squeeze(-1) # Global Max Pooling
        return self.fc(x)

model = TextClassifier(MAX_WORDS, 64, NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def train():
    for epoch in range(15):
        model.train()
        train_loss, correct = 0, 0
        for texts, labels in train_loader:
            texts, labels = texts.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(texts)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()

        # Валидация
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for texts, labels in val_loader:
                texts, labels = texts.to(DEVICE), labels.to(DEVICE)
                outputs = model(texts)
                val_correct += (outputs.argmax(1) == labels).sum().item()

        print(f"Эпоха {epoch+1} | Acc: {correct/len(train_loader.dataset):.4f} | Val Acc: {val_correct/len(val_loader.dataset):.4f}")

train()

Эпоха 1 | Acc: 0.1197 | Val Acc: 0.1654
Эпоха 2 | Acc: 0.2422 | Val Acc: 0.2432
Эпоха 3 | Acc: 0.3521 | Val Acc: 0.3160
Эпоха 4 | Acc: 0.4346 | Val Acc: 0.3696
Эпоха 5 | Acc: 0.4986 | Val Acc: 0.4022
Эпоха 6 | Acc: 0.5510 | Val Acc: 0.4328
Эпоха 7 | Acc: 0.5949 | Val Acc: 0.4615
Эпоха 8 | Acc: 0.6227 | Val Acc: 0.4769
Эпоха 9 | Acc: 0.6552 | Val Acc: 0.4878
Эпоха 10 | Acc: 0.6805 | Val Acc: 0.4931
Эпоха 11 | Acc: 0.7058 | Val Acc: 0.5055
Эпоха 12 | Acc: 0.7238 | Val Acc: 0.5212
Эпоха 13 | Acc: 0.7415 | Val Acc: 0.5309
Эпоха 14 | Acc: 0.7579 | Val Acc: 0.5465
Эпоха 15 | Acc: 0.7714 | Val Acc: 0.5421
